# Lecture 17: Trusting, reproducing and extending a solution

**The challenge.** Sixteen lectures produced extremals that satisfy the
necessary conditions to round-off. Satisfying the package's own equations
is not the same as being right. A sign error in a problem statement, a
missed switch, an integrator tolerance set too loose, or a second extremal
nobody looked for can all hide behind a residual of $10^{-12}$. Engineering
practice asks more: independent checks, comparison with the literature,
results others can reproduce and see, and a clear statement of what the
method cannot do. This last lecture is about that practice, and it maps every
part of the package to where it was taught, so nothing is left unexplained.

**After this lecture you can**

- verify an extremal by checks that do not share code with the solver;
- run and extend the benchmark suite, and record and render animated solves;
- find any function of the package and the lecture that explains it;
- work on the package itself, and say what is still open.

## 1. Benchmarks as a scientific habit

Numerical optimization learned to distrust its own successes early. In 1981
Willi Hock and Klaus Schittkowski published a collection of 119 test
problems for nonlinear programming, and for decades no new method was taken
seriously until it had been run on them. Jorge Moré, who with Burton Garbow
and Kenneth Hillstrom wrote MINPACK (the root finder under every shooting
solve here, Lecture 9), later assembled with Elizabeth Dolan and Todd Munson
the COPS collection of large optimization problems. That is where Lecture
11's robot arm comes from. Dolan and Moré's *performance profiles* (2002)
then changed how solvers are compared: not by a table of wins, but by the
fraction of problems each solves within a factor of the best time. The
package's `benchmarks/` follows the same habit: canonical problems, published
values, and a test that fails if a result drifts.

## 2. Checks that do not share code with the solver

Take the train between two stations of Lecture 14, solved automatically,
and check it five ways.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

from auto_optimal_control import examples
from auto_optimal_control.homotopy import Homotopy

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

V, D, umax = 2.0, 10.0, 0.5
h = Homotopy(examples.speed_limited_min_time())
r = h.solve([V, D, umax], ([-3, -3], [0, 3]), (1.0, 20.0))
shoot, sol = r.shooting, r.solution
tr = shoot.trajectory(sol.v, n_points=4001)

# 1. The package's own residual and diagnostics.
d = shoot.diagnostics(sol.v)
print(f"1. |R| = {sol.residual_norm:.1e}; mu on the arc {np.round(d['boundary_mu'][0], 6)}; "
      f"free-arc violation {d['free_arc_violation']}")
# 2. A conserved quantity: autonomous with a free final time, so H = 0 throughout.
#    On the boundary arc H is the boundary arc's Hamiltonian, at the control riding the limit.
kx, px = shoot.kernels, shoot.params
H = [kx.boundary_hamiltonian(c, ti, zi, px) if c >= 0 else kx.hamiltonian(ti, zi, px)
     for ti, zi, c in zip(tr["t"], tr["z"], tr["arc"])]
print(f"2. max |H| along the extremal: {np.abs(H).max():.1e}")
# 3. The closed form.
print(f"3. t_f = {sol.tf:.14f}, closed form D/V + V/u_max = {D / V + V / umax}")
# 4. An independent simulation of the original equations under the control history.
u_of_t = interp1d(tr["t"], tr["u"][:, 0], kind="previous", fill_value="extrapolate")
sim = solve_ivp(lambda t, y: [y[1], float(u_of_t(t))], (0.0, sol.tf), [0.0, 0.0],
                max_step=1e-3, rtol=1e-10, atol=1e-12)
print(f"4. simulated end state {np.round(sim.y[:, -1], 5)} (target [{D}, 0]); "
      f"max speed {sim.y[1].max():.6f} (limit {V})")
# 5. Another draw of the costates, another seed.
r2 = h.solve([V, D, umax], ([-3, -3], [0, 3]), (1.0, 20.0), seed=7)
print(f"5. seed 7: t_f = {r2.solution.tf:.14f}, structure {r2.arcs}")

Each check catches a different mistake. The residual and diagnostics catch
solver failures and wrong structures. Constancy of $H$ catches integration
error and a wrong derivation. The closed form, when there is one, catches
everything. The independent simulation uses none of the package's
equations, so it catches a mis-stated problem, provided the simulation is
written from the physics, not copied from the problem object. Another seed
probes for other extremals. For a problem without a closed form, compare
with published values, as the benchmarks do.

## 3. The benchmark suite

`benchmarks/problems.py` poses each problem, gives `Homotopy` the problem,
its parameters and a costate box, and nothing else, and compares with a
published or closed-form value. From a clone of the repository:

```bash
uv run python benchmarks/run.py                      # every benchmark, with a table
uv run python benchmarks/run.py goddard robot_arm    # some of them
uv run python benchmarks/run.py --markdown benchmarks/RESULTS.md
uv run pytest --runslow tests/test_benchmarks.py     # as tests (CI runs this)
```

Each benchmark is a function returning `Result` rows (`value`, `reference`,
`source`, `tolerance`, `details`). A new benchmark needs a problem, a
reference value with its source, and one call to `Homotopy.solve`; add it to
`ALL` and it is run and tested with the rest. The current suite covers the
brachistochrone, Bryson–Denham, velocity and speed limits, orbit raising,
Earth–Mars, the COPS arm, the cart-pole and Goddard with and without its
pressure limit. Direct methods appear only as comparisons, as in Lecture 9's
cart-pole.

## 4. Watching a solve

`benchmarks/animate.py` records a solve and renders every recorded iterate
as video: each Newton evaluation a real trajectory, each continuation step a
frame. It needs the `viz` dependency group, and ffmpeg for MP4 (GIF works
without):

```bash
uv run --group viz python benchmarks/animate.py goddard --stills
uv run --group viz python benchmarks/animate.py cart_pole --reuse --out outputs/cart_pole.gif
```

Each suite in `benchmarks/visuals/` has `record()` (run the solves, return
plain arrays), `save`/`load` (an `.npz`), `animate(data, path)` and a
`PROBLEM` card that opens the video. `--reuse` renders a saved recording
without solving again. The rendered videos and stills are in `docs/visuals/`:

![Goddard's rocket with the dynamic-pressure limit](https://raw.githubusercontent.com/idm23/auto-optimal-control/main/docs/visuals/goddard_q.png)

## 5. The whole package, and where each part was taught

| module | what it holds | lectures |
|---|---|---|
| `problem` | `OptimalControlProblem`, `Box`, `UnitVector`, `PathConstraint` | 1, 4, 13 |
| `pontryagin` | `derive` → `PontryaginSystem` (`hamiltonian_full`, `control_law`, `switching_functions`, `costate_dynamics`, `boundary`, `path_constraints`, `singular_arcs`), `constraint_derivatives` | 2–4, 12–14 |
| `codegen` | `generate_c`: the necessary conditions as C99 | 6 |
| `build` | `build_kernels`, `cache_root`, `Kernels` (`propagate`, `propagate_batch`, `trajectory`, `z0`, `write_c`, pointwise kernels, FNI objects) | 6–8 |
| `numeric` | `NumericSystem`: the lambdified reference | 6, 7 |
| `shooting` | `SingleShooting`, `MultiArcShooting` (`from_penalty`, `from_smoothed`, `diagnostics`), `continuation`, `parameter_continuation`, `ShootingError` | 9, 10, 12, 14 |
| `homotopy` | `regularize`, `exact`, `Homotopy`, `solve`, `HomotopyResult`, `Step` | 10, 15 |
| `examples` | textbook problems with known solutions | throughout |

A few entry points were mentioned but not exercised. Here they are, briefly.
`ArcDynamics` and `switching_events` are native FNI objects. With them, FNI's
own `solve_ivp` integrates a fixed arc and locates the switches, without
going through `propagate`:

In [ ]:
from fast_numerical_integration.integrators import solve_ivp as fni_solve_ivp

from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import constraint_derivatives, derive

k = build_kernels(derive(examples.double_integrator_min_time()))
p = [1.0, 1.0, 0.0]
z0 = k.z0([1.0, 1.0], p)
arc = k.initial_arc(0.0, z0, p)
res = fni_solve_ivp(k.ArcDynamics(p, arc), (0.0, 2.0), z0, method="DOP853",
                    events=k.switching_events(p), rtol=1e-12, atol=1e-12)
print("S = 0 found by FNI at t =", res.t_events[0], " (the arc does not flip here; propagate flips it)")
print("autonomous:", derive(examples.double_integrator_min_time()).is_autonomous)
print("constraint_derivatives:", constraint_derivatives(examples.bryson_denham(),
                                                        examples.bryson_denham().path_constraints[0].expr))
batch = k.propagate_batch(0.0, np.array([z0, k.z0([1.2, 1.1], p)]), 2.0, p, stm=True, n_threads=2)
print("propagate_batch with the STM:", batch["stm"].shape, " switches per row:", batch["n_switches"])

## 6. Working on the package

- **Tests.** `uv run pytest` runs the fast suite, compiling every example
  from scratch; `uv run pytest --runslow` adds the benchmarks, the
  animations and the slower automatic solves. The tests compare every
  kernel with the NumPy reference and with finite differences, every
  extremal with closed forms, and every derived law with a brute-force
  minimization (Lecture 4's check).
- **The kernel cache** is `~/.cache/auto-optimal-control`, or
  `$AUTO_OPTIMAL_CONTROL_CACHE`; delete it to force rebuilds, or pass
  `force=True`.
- **A local integrator.** To try changes to fast-numerical-integration
  without touching the lockfile:
  `uv run --with-editable ../fast-numerical-integration pytest`.
- **Plain C.** `Kernels.write_c(directory)` exports the generated header and
  source, which depend only on `<math.h>`, for use without Python.

## 7. What is still open

Being honest about limits is part of the method.

- **Global optimality.** Everything here is local and necessary (Lecture 5).
  Several extremals are common (Lectures 9 and 10). Searching costate space
  more cleverly, with gradient-free methods or learned samplers that propose
  costates, is the natural next layer. It is why the forward map
  (`propagate`, `propagate_batch`, `terminate`) is the core of the package and
  every solver is optional.
- **Several limits at once.** Exact arcs handle one active limit, or one
  singular control, at a time, and one limit with a single bang-bang
  control. Real robots ride several limits simultaneously, which needs arcs
  defined by an *active set* of constraints and bounds.
- **High-order limits.** Order 3 and up chatter (Lecture 14); reformulation
  is the practical answer.
- **Long horizons.** Single shooting's sensitivity grows with the
  trajectory (Lecture 8). Multiple shooting, which splits the interval, tames
  it, and is a planned layer.

## Summary

- Trust a result through checks that do not share code with the solver:
  diagnostics, conserved quantities, closed forms, independent simulation,
  other seeds, the literature.
- `benchmarks/run.py` and `tests/` reproduce every published comparison;
  `benchmarks/animate.py` records and renders solves.
- Every module and function of the package has a lecture; section 5 is the
  map.

## Exercises (projects)

1. Add a benchmark: Zermelo's navigation problem with a closed-form
   reference (Bryson and Ho, section 2.7), as a function in
   `benchmarks/problems.py` returning `Result` rows.
2. Take the cart-pole of Lecture 9 to minimum time with $|u| \le 20$ N, solve
   it with `Homotopy`, and verify it with all five checks of section 2.
3. Pose a two-joint jerk-limited robot with velocity limits on both joints.
   Which parts of the package solve it today, and where does it stop? Write
   down the active-set structure the answer needs.

## Further reading

- W. Hock and K. Schittkowski, *Test Examples for Nonlinear Programming
  Codes* (Springer, 1981).
- E. D. Dolan and J. J. Moré, "Benchmarking optimization software with
  performance profiles", *Mathematical Programming* 91, 2002.
- J. T. Betts, *Practical Methods for Optimal Control Using Nonlinear
  Programming* (SIAM, 2010), for the direct methods this package compares
  with but does not use.
- H. J. Pesch, "A practical guide to the solution of real-life optimal
  control problems", *Control and Cybernetics* 23, 1994.